# Analisi S-value — decadimento del Cu-64 nel nucleo cellulare

Notebook per aggregare i batch della simulazione Geant4 (`S-value`) e ricavare:
- l'S-value combinato per nucleo e citoplasma, con errore statistico corretto sulla media;
- la distribuzione $f_1(z_n)$ dell'energia specifica per singolo decadimento nel nucleo (istogramma 6), utile per il modello LQ e come punto di partenza per il modello MK;
- un controllo incrociato tra i valori riportati nei file `s_batchNN.txt` e quelli ricalcolati dagli istogrammi nei file `.root`.

**Nota importante.** I file `s_batchNN.txt` sono aperti in modalità *append* (`Run::EndOfRun`, `fopen(..., "a")`): se un batch è stato eseguito più volte, il file contiene più righe. I file `.root`, invece, vengono sovrascritti a ogni esecuzione (`analysisManager->OpenFile()` di default). Per coerenza, da ogni `.txt` viene quindi usata **solo l'ultima riga**, corrispondente al contenuto attuale del `.root`.

**Limite noto.** Gli istogrammi qui disponibili danno $f_1(z_n)$ sull'intero nucleo, sufficiente per il modello LQ e per l'S-value. Il modello MK richiederebbe $f_1(z)$ in domini sub-nucleari di raggio $r_d \ll R_n$: non essendo disponibile uno scoring per posizione, in questo notebook si userà come approssimazione $r_d = R_n$ (dominio = intero nucleo), da sostituire quando sarà disponibile uno scoring posizionale.


## 1. Setup


In [ ]:
import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot

# Cartella con i file di output (adattare se necessario)
BUILD_DIR = Path("../build")

root_files = sorted(BUILD_DIR.glob("cu64_batch*.root"))
txt_files  = sorted(BUILD_DIR.glob("s_batch*.txt"))

print(f"Trovati {len(root_files)} file .root e {len(txt_files)} file .txt")
for f in root_files:
    print(" ", f.name)


## 2. Lettura dei file `.txt` (ultima riga per batch)

Colonne, nell'ordine scritto da `Run::EndOfRun()`:

```
R_nucleo[nm]  spessore_citoplasma[nm]  E_cin_primario[eV]  S_cito[mGy/Bq.s]  err_S_cito  S_nucl[mGy/Bq.s]  err_S_nucl
```

`err_S_cito`/`err_S_nucl` sono la deviazione standard della distribuzione a singolo decadimento (sigma di $f_1(z)$), non l'errore sulla media.


In [ ]:
TXT_COLS = [
    "R_nucl_nm", "cyto_thick_nm", "Ekin_eV",
    "S_cyto_mGy_Bqs", "sigma_S_cyto",
    "S_nucl_mGy_Bqs", "sigma_S_nucl",
]

def batch_id_from_name(path, pattern):
    m = re.search(pattern, path.name)
    return m.group(1) if m else path.stem

txt_rows = {}
n_lines_per_batch = {}

for f in txt_files:
    bid = batch_id_from_name(f, r"s_(batch\d+)")
    df = pd.read_csv(f, sep=r"\s+", header=None, names=TXT_COLS)
    n_lines_per_batch[bid] = len(df)
    txt_rows[bid] = df.iloc[-1]  # solo l'ultima riga: corrisponde al .root attuale

txt_summary = pd.DataFrame(txt_rows).T
txt_summary.index.name = "batch"
txt_summary["n_run_lines"] = pd.Series(n_lines_per_batch)

print("Numero di righe (= numero di esecuzioni) per ciascun batch:")
print(txt_summary["n_run_lines"])
print()
txt_summary


Se qualche batch ha più di una riga, vale la pena controllare a mano se le righe scartate sono davvero duplicati/test falliti, o se contengono statistica che vorresti recuperare (nel qual caso il seed andrebbe randomizzato e i run rilanciati).


## 3. Lettura dei file `.root`

Gli istogrammi sono `TH1D` con ID fissi:

| ID | Contenuto |
|----|-----------|
| 0  | energia cinetica e- Auger |
| 1  | energia cinetica e- Auger a bassa energia |
| 5  | energia depositata nel citoplasma |
| 6  | energia depositata nel nucleo |
| 7  | lunghezza di traccia della primaria |
| 10 | lunghezza di traccia dei secondari carichi |

Qui usiamo l'istogramma **6** (nucleo), che è quello rilevante per l'S-value e per $f_1(z_n)$.


In [ ]:
NUCL_HIST_ID = "6"

def read_hist(root_path, hist_id):
    with uproot.open(root_path) as f:
        h = f[f"{hist_id};1"]
        counts, edges = h.to_numpy()
        return counts, edges

batch_hists = {}   # bid -> (counts, edges)
batch_n_events = {}  # bid -> numero di eventi (somma dei conteggi dell'istogramma)

for f in root_files:
    bid = batch_id_from_name(f, r"cu64_(batch\d+)")
    counts, edges = read_hist(f, NUCL_HIST_ID)
    batch_hists[bid] = (counts, edges)
    batch_n_events[bid] = counts.sum()

n_events_summary = pd.Series(batch_n_events, name="N_eventi_da_istogramma")
n_events_summary.index.name = "batch"
n_events_summary


**Controllo:** se `N_eventi_da_istogramma` non è uguale per tutti i batch, vuol dire che sono stati lanciati con statistiche diverse (va bene, ma la combinazione qui sotto ne tiene conto pesando per $N$); se invece tutti i bin sono a zero per qualche batch, quel `.root` va controllato a parte (probabile run vuoto o fallito).


## 4. S-value per batch: media e deviazione standard dall'istogramma

Ricalcoliamo media e sigma della distribuzione a singolo decadimento direttamente dall'istogramma, per confronto con i valori riportati nel `.txt`. Nota: qui l'energia è in **keV** (bin edges 0–700 keV secondo il binning dichiarato); adattare `EDEP_UNIT_keV` se nel tuo `EventAction` usi un'altra unità di conversione per `FillH1`.


In [ ]:
EDEP_UNIT_keV = 1.0  # se FillH1(6, edep/keV) -> 1.0; se invece è edep/MeV -> cambiare qui

def hist_mean_std(counts, edges):
    centers = 0.5 * (edges[:-1] + edges[1:])
    n = counts.sum()
    if n == 0:
        return 0.0, 0.0, 0
    mean = np.sum(counts * centers) / n
    var = np.sum(counts * centers**2) / n - mean**2
    std = np.sqrt(var) if var > 0 else 0.0
    return mean * EDEP_UNIT_keV, std * EDEP_UNIT_keV, int(n)

hist_summary = {}
for bid, (counts, edges) in batch_hists.items():
    mean_keV, std_keV, n = hist_mean_std(counts, edges)
    hist_summary[bid] = {"mean_edep_keV": mean_keV, "sigma_edep_keV": std_keV, "N_eventi": n}

hist_summary = pd.DataFrame(hist_summary).T
hist_summary.index.name = "batch"
hist_summary


## 5. Massa del nucleo e conversione in S-value (Gy/decadimento)

Nucleo modellato come sfera d'acqua di raggio `R_nucl_nm` (dal `.txt`). Se nel tuo `DetectorConstruction` la geometria è diversa (es. non è una sfera, o la densità non è quella dell'acqua), **aggiorna questa cella** di conseguenza.


In [ ]:
RHO_WATER_kg_m3 = 1000.0

def nucleus_mass_kg(R_nm):
    R_m = R_nm * 1e-9
    volume_m3 = (4.0 / 3.0) * np.pi * R_m**3
    return RHO_WATER_kg_m3 * volume_m3

# Raggio nucleo: assunto uguale per tutti i batch, preso dal primo txt disponibile
R_nucl_nm = txt_summary["R_nucl_nm"].iloc[0]
if not np.allclose(txt_summary["R_nucl_nm"], R_nucl_nm):
    print("ATTENZIONE: i batch non hanno tutti lo stesso raggio del nucleo!")
    print(txt_summary["R_nucl_nm"])

m_nucl_kg = nucleus_mass_kg(R_nucl_nm)
print(f"Raggio nucleo: {R_nucl_nm} nm  ->  massa nucleo: {m_nucl_kg:.4e} kg")

# Conversione energia depositata (keV) -> dose (Gy) per decadimento
keV_to_J = 1.602176634e-16

hist_summary["S_nucl_from_hist_Gy_per_decay"] = (
    hist_summary["mean_edep_keV"] * keV_to_J / m_nucl_kg
)
hist_summary["sigma_S_nucl_from_hist_Gy_per_decay"] = (
    hist_summary["sigma_edep_keV"] * keV_to_J / m_nucl_kg
)
hist_summary


## 6. Confronto tra `.txt` e `.root`

Confrontiamo `S_nucl` riportato nel `.txt` (già in mGy/Bq·s, quindi da dividere per 1000 per avere Gy) con `S_nucl_from_hist_Gy_per_decay` ricalcolato qui sopra. Se i due non coincidono (a meno di piccole differenze numeriche), va ricontrollata l'unità usata in `FillH1(6, ...)` (parametro `EDEP_UNIT_keV` sopra) o la definizione di massa del nucleo in `DetectorConstruction`.


In [ ]:
comparison = pd.DataFrame({
    "S_nucl_txt_Gy_per_decay": txt_summary["S_nucl_mGy_Bqs"].astype(float) / 1000.0,
    "S_nucl_hist_Gy_per_decay": hist_summary["S_nucl_from_hist_Gy_per_decay"],
})
comparison["rel_diff_%"] = 100 * (comparison["S_nucl_hist_Gy_per_decay"]
                                   - comparison["S_nucl_txt_Gy_per_decay"]) / comparison["S_nucl_txt_Gy_per_decay"]
comparison


## 7. Combinazione dei 10 batch

Media pesata sui decadimenti totali (non media semplice dei batch), con errore statistico sulla media (deviazione standard della distribuzione a singolo decadimento, divisa per $\sqrt{N_\mathrm{tot}}$) e, separatamente, la dispersione batch-a-batch come stima della sistematica tra run.


In [ ]:
N = hist_summary["N_eventi"].astype(float)
S = hist_summary["S_nucl_from_hist_Gy_per_decay"]
sigma = hist_summary["sigma_S_nucl_from_hist_Gy_per_decay"]

N_tot = N.sum()

# Media pesata sui decadimenti totali (equivalente a sommare tutti gli eventi insieme)
S_combined = np.sum(N * S) / N_tot

# Errore statistico sulla media: sigma della distribuzione singolo-decadimento / sqrt(N_tot)
# Si combina la sigma per-batch pesata sugli eventi
sigma_pooled = np.sqrt(np.sum(N * sigma**2) / N_tot)
err_stat = sigma_pooled / np.sqrt(N_tot)

# Dispersione batch-a-batch (sistematica tra run), solo se piu' di un batch
S_per_batch = S.values
std_between_batches = S_per_batch.std(ddof=1) if len(S_per_batch) > 1 else 0.0

print(f"Numero totale di decadimenti simulati: {N_tot:.0f}")
print(f"S-value nucleo combinato:        {S_combined:.4e} Gy/decadimento")
print(f"Errore statistico sulla media:    {err_stat:.4e} Gy/decadimento")
print(f"Dispersione batch-a-batch (std):  {std_between_batches:.4e} Gy/decadimento")


In [ ]:
S_per_batch = hist_summary["S_nucl_from_hist_Gy_per_decay"].values
N_per_batch = hist_summary["N_eventi"].values

# sigma atteso per ciascun batch, dalla sigma "pooled" a singolo decadimento
sigma_expected_per_batch = sigma_pooled / np.sqrt(N_per_batch)

chi2 = np.sum((S_per_batch - S_combined)**2 / sigma_expected_per_batch**2)
dof = len(S_per_batch) - 1
print(f"chi2/dof = {chi2/dof:.2f}  (dof = {dof})")

## 8. Distribuzione combinata $f_1(z_n)$

Si sommano i conteggi dei 10 istogrammi (stesso binning) per ottenere la distribuzione a singolo decadimento con tutta la statistica disponibile, poi la si converte in energia specifica $z_n$ (Gy/decadimento) usando la massa del nucleo.


In [ ]:
# Verifica che tutti gli istogrammi abbiano lo stesso binning
edges_ref = None
counts_sum = None
for bid, (counts, edges) in batch_hists.items():
    if edges_ref is None:
        edges_ref = edges
        counts_sum = counts.copy().astype(float)
    else:
        if not np.allclose(edges, edges_ref):
            raise ValueError(f"Binning diverso nel batch {bid}: impossibile sommare direttamente.")
        counts_sum += counts

centers_keV = 0.5 * (edges_ref[:-1] + edges_ref[1:]) * EDEP_UNIT_keV
z_n_Gy = centers_keV * keV_to_J / m_nucl_kg

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(z_n_Gy, counts_sum, width=np.diff(z_n_Gy).mean(), align="center")
ax.set_xlabel(r"$z_n$ [Gy / decadimento]")
ax.set_ylabel("conteggi (tutti i batch)")
ax.set_title(r"Distribuzione $f_1(z_n)$ — energia specifica nel nucleo per singolo decadimento")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

print(f"Eventi totali nell'istogramma combinato: {counts_sum.sum():.0f}")
print(f"Media della distribuzione combinata:      {np.sum(counts_sum * z_n_Gy) / counts_sum.sum():.4e} Gy/decadimento")


## 9. Prossimi passi

- **Modello LQ**: con `S_combined` (S-value per decadimento) e il numero di decadimenti $\tilde N$, si ottiene $D = \tilde N \cdot S_\mathrm{combined}$ e quindi $S(D)$ dall'Eq. LQ del capitolo.
- **Modello MK**: la distribuzione qui ottenuta è $f_1(z_n)$ sull'**intero nucleo**, non nei domini sub-nucleari $r_d$. Per il vero calcolo MK serve uno scoring per posizione (es. un ntuple con posizione + energia di ogni deposito), da cui ricostruire $f_1(z)$ per domini sferici di raggio arbitrario in post-processing, come descritto nel capitolo. Finché non è disponibile, si può usare $r_d = R_n$ come approssimazione (dominio = nucleo), sapendo che sovrastima l'effetto di saturazione rispetto a domini più piccoli.
- **Numero di radionuclidi vs segnale PET**: una volta scelti $\alpha_0,\beta,y_0,r_d$ per la linea cellulare, si può invertire il modello come nella Sez. 3.7 del capitolo usando `S_combined` al posto di $\langle z_n\rangle$ calcolato da simulazioni a granularità di dominio.
